In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DatasetPath = "/content/drive/MyDrive/dataset/brain-tumor-mri-dataset"

In [ ]:
import os
print(os.listdir(DatasetPath))

In [ ]:
Testing_data_path = "/content/drive/MyDrive/dataset/brain-tumor-mri-dataset/Testing"
Training_data_path = "/content/drive/MyDrive/dataset/brain-tumor-mri-dataset/Training"

In [ ]:
import tensorflow as tf
Train_data = tf.keras.utils.image_dataset_from_directory(
    Training_data_path,
    image_size = (256,256),
    batch_size = 64,
    shuffle = True
)

Test_data = tf.keras.utils.image_dataset_from_directory(
    Testing_data_path,
    image_size = (256,256),
    batch_size = 64,
    shuffle = True
)

In [ ]:
Train_data = (
    Train_data
    .cache()
    .shuffle(1000)
    .prefetch(AUTOTUNE)
)

Test_data = (
    Test_data
    .cache()
    .prefetch(AUTOTUNE)
)

In [ ]:
normalization = tf.keras.layers.Rescaling(1./225)

In [ ]:
Train_data = Train_data.map(
    lambda x,y : (normalization(x),y)
)

Test_data = Test_data.map(
    lambda x,y : (normalization(x),y)
)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(256, 256, 3)),

    tf.keras.layers.Rescaling(1./255),

    data_augmentation,

    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        256,
        (3,3),
        activation='relu',
        padding='same',
        name='last_conv'
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(4, activation='softmax')
])

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    Train_data,
    validation_data=Test_data,
    epochs=30,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)